In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
import numpy as np
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
import pandas as pd

books = pd.read_csv("books_Cleaned.csv")

In [4]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [5]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: object

In [6]:
books["tagged_description"].to_csv("tagged_description.txt", index=False, header=False, sep="\n")

In [7]:
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of size 1189, which is longer than the specified 1000
Created a chunk of size 1267, which is longer than the specified 1000
Created a chunk of size 2010, which is longer than the specified 1000
Created a chunk of size 1225, which is longer than the specified 1000
Created a chunk of size 1184, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1057, which is longer than the specified 1000
Created a chunk of size 1270, which is longer than the specified 1000
Created a chunk of size 1635, which is longer than the specified 1000
Created a chunk of size 1132, which is longer than the specified 1000
Created a chunk of s

In [8]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [9]:
"""
Initial TFIDF EMBEDDING
"""


# class TfidfEmbeddings(Embeddings):
#     def __init__(self):
#         self.vectorizer = TfidfVectorizer(
#             max_features=5000,
#             stop_words="english"
#         )
#         self._fit_done = False
#
#     def fit(self, texts):
#         self.vectorizer.fit(texts)
#         self._fit_done = True
#         return self
#
#     def embed_documents(self, texts):
#         if not self._fit_done:
#             self.fit(texts)
#         matrix = self.vectorizer.transform(texts)
#         return matrix.toarray().astype(np.float32).tolist()
#
#     def embed_query(self, text):
#         if not self._fit_done:
#             raise ValueError("Embedding model must be fit on documents before querying.")
#         vector = self.vectorizer.transform([text])
#         return vector.toarray().astype(np.float32)[0].tolist()
#
#
# # Build a local embedding model from your documents
# texts = [doc.page_content for doc in documents]
# embedding_model = TfidfEmbeddings().fit(texts)
#
# db_books = Chroma.from_documents(
#     documents=documents,
#     embedding=embedding_model
# )
#


'\nInitial TFIDF EMBEDDING\n'

In [10]:
"""
OPENAI EMBEDDING (NEED TO USE API KEY)
"""
#db_books = Chroma.from_documents(
#    documents=documents,
#    embedding=OpenAIEmbeddings()
#)

'\nOPENAI EMBEDDING (NEED TO USE API KEY)\n'

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db_books = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model
)

C:\Users\Arup sarkar\.conda\envs\condaenv3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [12]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)

In [13]:
docs

[Document(id='75be3ef6-2e20-4cc4-9bd5-ff8892a869fb', metadata={'source': 'tagged_description.txt'}, page_content="9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.\n9780786808373 Introducing your baby to birds, cats, dogs, and babies through fine art, illsutration and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing baby to some basic -- and sometimes playful -- information on the subjects."),
 Document(id='1371ddcb-d03e-48b5-b4d5-d4e5484ca8b8', metadata={'source': 'tagged_description.txt'}, page_content="9780064402453 ‘Racso, a brash and boastful little rodent, is making his way to Thorn

In [14]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [15]:
def retrieve_semantic_recommendations(query: str, top_k: int = 50) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)

    books_list = []

    for doc in recs:
        lines = doc.page_content.strip('"').split("\n")
        for line in lines:
            line = line.strip()
            if line:
                try:
                    isbn = int(line.split()[0])
                    books_list.append(isbn)
                except:
                    continue

    return books[books["isbn13"].isin(books_list)].head(top_k)

In [16]:
retrieve_semantic_recommendations("A book to teach children about nature",top_k=50)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,Joe and his parents are enjoying a summer holi...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 Joe and his parents are enjoying...
59,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,The Family Way,9780007151240 It should be the most natural th...
60,9780007151677,0007151675,Endless Night,Agatha Christie,FICTION,http://books.google.com/books/content?id=kY1wu...,Gipsy S Acre Was A Truly Beautiful Upland Site...,2002.0,3.77,303.0,12740.0,Endless Night,9780007151677 Gipsy S Acre Was A Truly Beautif...
61,9780007153589,0007153589,How to be Alone,Jonathan Franzen,Literary Collections,http://books.google.com/books/content?id=ozVWa...,'The Harper's Essay' is reprinted in this volu...,2004.0,3.60,306.0,327.0,How to be Alone,9780007153589 'The Harper's Essay' is reprinte...
383,9780061144899,0061144894,When the Heart Waits,Sue Monk Kidd,Religion,http://books.google.com/books/content?id=JlP91...,From the Bestselling Author of The Secret Life...,2006.0,4.17,240.0,2141.0,When the Heart Waits: Spiritual Direction for ...,9780061144899 From the Bestselling Author of T...
392,9780061208492,0061208493,The Complete C. S. Lewis Signature Classics,C. S. Lewis,Religion,http://books.google.com/books/content?id=JaC0_...,Seven Spiritual Masterworks by C. S. Lewis Thi...,2007.0,4.61,746.0,873.0,The Complete C. S. Lewis Signature Classics,9780061208492 Seven Spiritual Masterworks by C...
393,9780061238239,0061238236,The End of Days,Zecharia Sitchin,History,http://books.google.com/books/content?id=EIBlj...,A conclusion to the Earth Chronicles series br...,2007.0,4.06,336.0,470.0,The End of Days: Armageddon and Prophecies of ...,9780061238239 A conclusion to the Earth Chroni...
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
406,9780064403870,0064403874,"R-T, Margaret, and the Rats of NIMH",Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=WTHHH...,"When Margaret and her younger brother, Artie, ...",1991.0,3.52,272.0,631.0,"R-T, Margaret, and the Rats of NIMH",9780064403870 When Margaret and her younger br...
407,9780064404419,0064404412,The Rainbow People,Laurence Yep,Juvenile Fiction,http://books.google.com/books/content?id=5AHwq...,"""Culled from 69 stories collected in a [1930s]...",1992.0,3.75,208.0,202.0,The Rainbow People,"9780064404419 ""Culled from 69 stories collecte..."


In [17]:
#recs = db_books.similarity_search(query, k=50)
#print(len(recs))  # how many chunks returned
#for doc in recs[:3]:
#    print(doc.page_content[:200])
#    print("---")

In [18]:
#recs = db_books.similarity_search(query, k=50)

#books_list = []
#for doc in recs:
#    lines = doc.page_content.strip('"').split("\n")
#    for line in lines:
#        line = line.strip()
#        if line:
#            try:
#                isbn = int(line.split()[0])
#                books_list.append(isbn)
#            except:
                #continue

#print(len(books_list))         # total ISBNs extracted
#print(len(set(books_list)))    # unique ISBNs

In [19]:
#print(type(books_list[0]))      # what type are the extracted ISBNs
#print(books["isbn13"].dtype)    # what dtype is the column
#print(books["isbn13"].head())   # see actual values